« model_20 — AKRABALIK EĞİTİMİ (train_kinship) · COLAB KOŞULARI »

**Adım 4: dış not defteri, 32 aile (192 kişi).** Eğitimde yalnız 1R cümleleri ve ara adımlı 2R (`<steps>`); kısa 2R, adlı ilişkiler ve 3R yok. Sınav: 1R_T, 2R_T (eğitimde), 2R_UT (hiç görülmemiş). Kullanıcı, 27 Eylül: *"testleri colab taşıyalım çünkü artık bilgisayarda birşeyleri koşmak zamansal olarak maliyetli oldu"*.
Model: Model X (`deneme2/model_20/model_20.py`, varsayılan ayarlar; diyagram `model_x_20.html`). Bu klasör yalnız akrabalık eğitimine özgü: veri `data_20.py`, sınav `exam_kinship.py`, çalıştırıcı `colab_kinship.py`.
Karar kuralı ve sonuçlar: `train_kinship/OLCULENLER_kinship.md` (git dışı; eğitimler arası kıyas `deneme2/model_20/OLCULENLER_20.md`).

| hücre | iş |
|---|---|
| 0 HAZIRLIK | Drive, kod (GitHub'dan taze), veri (koddan; iz sabitle karşılaştırılır) |
| 1 KOŞU | GPU kapısı; eğitim arka planda (`colab_kinship.start`); `STEPS` kullanıcı kararıyla yazılır |
| 2 NABIZ | durum ve son sınav satırları |
| 3 SONUÇ | son ölçüm (üç koşul) ve modelin yazdıkları (kural 12) |
| X / Y / Z | durdur · Drive'daki kayıt · GPU durumu |

Veri koddan üretilir: `data_20.build` rastgelelik içermez, milisaniyeler sürer; iz (`IZ`) tutmazsa hücre durur (kural 9'un amacı: iki koşu aynı veriyi görsün).

In [ ]:
# 0 HAZIRLIK  |  CPU  |  tekrar: GUVENLI
# Cekirdek dustuyse ONCE bu hucre.
import os, sys, subprocess

# Canli kosu varken moduller yeniden yuklenirse kosu listesi sifirlanir.
for _mod in ('colab_20', 'colab_kinship'):
    if _mod in sys.modules:
        _live = [n for n, r in sys.modules[_mod].RUNS.items() if r['thread'].is_alive()]
        assert not _live, 'CEKIRDEK CANLI, kosu suruyor: %s -- HAZIRLIK gerekmiyor, 2 NABIZ' % _live

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/model_20'
os.makedirs(ROOT, exist_ok=True)

# Kod her seferinde TAZE cekilir -- Colab'da elle duzenleme birikmesin.
REPO = '/content/sekerai'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/sekerahmet/sekerai.git', REPO], check=True)
SRC = REPO + '/deneme2/model_20'                  # Model X ve genel egitim
KIN = SRC + '/train_kinship'                      # akrabalik egitimi: veri, sinav, calistirici
CODE = subprocess.run(['git', '-C', REPO, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('kod   ' + CODE)
assert os.path.exists(KIN + '/colab_kinship.py'), 'depoda train_kinship/colab_kinship YOK -- yerelde git push gerekli'
for _p in (SRC, KIN):
    if _p not in sys.path:
        sys.path.insert(0, _p)
for _m in ('data_20', 'model_20', 'model_20_transformer', 'train_20', 'exam_kinship', 'colab_kinship', 'colab_20'):
    sys.modules.pop(_m, None)          # taze kod gercekten yuklensin

import torch
import data_20
import train_20 as TR
import exam_kinship as EK
import colab_kinship as C

# Veri koddan; iz sabitle karsilastirilir.  Veri degisirse IZ BILEREK guncellenir.
STEP_ANSWERS = True
STEP_MARKER = True               # False: 2R sorusunda "<steps>" isareti yok (kullanici, 27 Eylul: "Steps bence aşırı zararlı şu an")
LONG_1R = True                   # True: 1R cevabi tam cumle, "?"dan sonra hep ozne (kullanici, 27 Eylul)
IZ = {(True, False): '7ae5615623aa', (False, False): '3898f9b3d551', (True, True): '240bdd2aaebf',
      (False, True): 'c87cdf8717c2'}[(STEP_MARKER, LONG_1R)]
DATA = data_20.build(step_answers=STEP_ANSWERS, step_marker=STEP_MARKER, long_1r=LONG_1R)
data_20.audit(DATA)
assert DATA['fingerprint'] == IZ, 'veri izi %s, beklenen %s -- defter eski' % (DATA['fingerprint'], IZ)
print('veri  iz %s  egitim %d cumle  sozluk %d' % (DATA['fingerprint'], len(DATA['train']), len(DATA['vocab'])))

In [ ]:
# 1 KOSU -- Model X  |  GPU  |  tekrar: RESUME=False iken ayni adla ikinci kez calisirsa eskisini
#     <ad>_eski_<zaman>/ klasorune TASIR, SILMEZ; RESUME=True iken son checkpoint'ten SURDURUR
SEED = 1                    # tohum 0: shared_steps_long1r_rope_s0; 1-3: modelx_s1..s3 (kullanici, 27 Eylul: "model X 3 thoum koş")
NAME = 'modelx_s%d' % SEED  # Model X: model_20 varsayilanlari (tek blok, 2 tur, akis normlu, L2, RoPE)
STEPS = 4000                # kullanici, 27 Eylul: "adım olarak da ilk olarak 4.000" (full batch: 1 adim = 1 epoch)
EVERY = 100                 # her 100 adimda sinav: kisa bicim + <steps> bicimi
LAYER_NORM = False          # True: L2 yerine LayerNorm (Model X'te fark etmedi: modelx_ln_s0 187)
SAVE_EVERY = 500            # kullanici, 27 Eylul: "yedek her 500 de olsun"; OUT/checkpoint_tNNNNN.pt (model + Adam + adim)
RESUME = False              # True: kosu kesildiyse (cekirdek dustu) once HAZIRLIK, sonra bu hucre -> son paketten devam

# --- GPU KAPISI (CLAUDE.md kural 2)
import torch
assert torch.cuda.is_available(), 'GPU YOK -- Runtime > Change runtime type'
_bos = torch.cuda.mem_get_info()[0] / 1e9
assert _bos > 2.0, "GPU'da sadece %.1f GB bos" % _bos
print('GPU kapisi GECTI: %s  bos %.1f GB' % (torch.cuda.get_device_name(0), _bos))
assert STEPS, 'STEPS yazilmadi -- adim sayisi kullanici karariyla'
assert DATA['fingerprint'] == '240bdd2aaebf', 'Model X kosulari uzun 1R + isaretli veriyle'

# ARKA PLANDA baslar, hucre HEMEN doner (kural 8).  Ilerleme: 2 NABIZ
OUT = ROOT + '/' + NAME
C.start(NAME, DATA, OUT, steps=STEPS, seed=SEED, every=EVERY, device='cuda', setting='shared',
        layer_norm=LAYER_NORM, save_every=SAVE_EVERY, resume=RESUME)
# compile varsayilan ACIK (kullanici, 27 Eylul: "bu sabit ayar ve yes olsun"); tarif varsayilani Muon + WSD (train_20)
print('basladi: %s  Model X  tohum %d  %d adim  LayerNorm %s  ->  %s' % (NAME, SEED, STEPS, LAYER_NORM, OUT))

In [ ]:
# 2 NABIZ  |  CPU  |  tekrar: GUVENLI
C.pulse(5)

In [ ]:
# 3 SONUC  |  CPU  |  tekrar: GUVENLI
# SAYI + METIN (kural 12): uc kosul (model yazar / ozne verilir / ilk cumle verilir) ve modelin yazdiklari.
import json
OUT = ROOT + '/' + NAME
_f = json.load(open(OUT + '/final.json', encoding='utf-8'))
for _k, _v in _f.items():
    print('%-26s %s' % (_k, '  '.join('%s %d' % kv for kv in _v['counts'].items())))
for _k in ('2R_UT_given0', '2R_UT_given2', '2R_UT_given8'):
    print('\n--', _k)
    for _q, _said, _want in _f[_k]['rows'][::6][:8]:          # her kisiden bir soru, 8 farkli kisi
        print('  %s\n     model: %s\n     dogru: %s' % (_q, _said, _want))

In [ ]:
# X DURDUR  |  CPU  |  tekrar: GUVENLI
# Bayrak koyar; iplik bir sonraki sinavda modeli Drive'a kaydedip cikar.
C.stop()

In [ ]:
# Y DRIVE'DAKI KAYIT  |  CPU  |  tekrar: GUVENLI
for _n in sorted(os.listdir(ROOT)):
    _d = ROOT + '/' + _n
    if os.path.isdir(_d):
        _fs = sorted(os.listdir(_d))
        print('%-28s %s' % (_n, ' '.join(_fs)))

In [ ]:
# Z GPU DURUMU  |  CPU  |  tekrar: GUVENLI
!nvidia-smi